# SnowPro Core (COF-C03) Study Guide
**Based on Tasty Bytes Vignette Labs + Official Exam Guide**

| Domain | Weight |
|--------|--------|
| 1.0 Features & Architecture | 31% |
| 2.0 Account Management & Governance | 20% |
| 3.0 Data Loading & Connectivity | 18% |
| 4.0 Performance & Transformation | 21% |
| 5.0 Data Collaboration | 10% |

> Concepts tagged with **(V1)** through **(V4)** reference the vignette file they come from. Concepts tagged **(Gap)** are not in the vignettes but appear on the exam.

---
# Domain 1.0 — Features & Architecture (31%)

## 1.1 Three-Layer Architecture

Snowflake's unique architecture separates compute, storage, and cloud services:

| Layer | Purpose | Key Details |
|-------|---------|-------------|
| **Cloud Services** | Brain — authentication, metadata, query optimization, result cache | Runs 24/7, charges only if >10% of daily compute |
| **Compute (Virtual Warehouses)** | Muscle — executes queries | Independent, scalable clusters; no shared state between warehouses |
| **Storage** | Columnar micro-partitions, compressed, encrypted | Managed by Snowflake; stored in cloud provider's blob storage (S3/Azure Blob/GCS) |

**Exam tip:** The query result cache lives in the **Cloud Services layer**, NOT in the warehouse. That's why results can be returned without resuming a suspended warehouse. **(V1)**

## 1.2 Snowflake Editions

| Edition | Key Features Added |
|---------|-------------------|
| **Standard** | Full SQL, Time Travel (1 day), secure data sharing, all base features |
| **Enterprise** | Multi-cluster warehouses, 90-day Time Travel, masking policies, row access policies, classification profiles, materialized views, search optimization, column-level security |
| **Business Critical** | HIPAA/PCI/FedRAMP compliance, Tri-Secret Secure, failover/failback, private connectivity |
| **VPS** | Virtual Private Snowflake — dedicated, isolated environment |

**Vignette lesson:** The `SNOWFLAKE.DATA_PRIVACY.CLASSIFICATION_PROFILE` and `SYSTEM$CLASSIFY` features failed on the trial account because they require **Enterprise Edition**. **(V4)**

**Exam tip:** Know which features require which edition. Materialized views, multi-cluster warehouses, and data masking = Enterprise. Private connectivity and Tri-Secret Secure = Business Critical.

## 1.3 Object Hierarchy

```
Organization
 └── Account
      ├── User / Role / Warehouse / Resource Monitor / Budget
      └── Database
           └── Schema
                ├── Table (permanent, transient, temporary, external, dynamic, Iceberg)
                ├── View (standard, secure, materialized)
                ├── Stage (internal, external)
                ├── File Format
                ├── Sequence
                ├── Stream
                ├── Task
                ├── Pipe (Snowpipe)
                ├── UDF / Stored Procedure
                ├── Tag
                ├── Masking Policy / Row Access Policy
                └── Data Metric Function (DMF)
```

**Key from vignettes:** Databases contain schemas, schemas contain everything else. Setting context (`USE DATABASE`, `USE SCHEMA`) determines the default resolution for unqualified names. **(V1–V4)**

## 1.4 Virtual Warehouses

### Sizing & Properties **(V1)**

| Parameter | Purpose | Default |
|-----------|---------|--------|
| `WAREHOUSE_SIZE` | XSmall → 6XLarge (each step 2x credits/hr) | XSmall |
| `AUTO_SUSPEND` | Seconds of inactivity before suspending | 600 (10 min) |
| `AUTO_RESUME` | Resume automatically on query | TRUE |
| `INITIALLY_SUSPENDED` | Start in suspended state | — |
| `WAREHOUSE_TYPE` | `STANDARD` or `SNOWPARK_OPTIMIZED` (16x memory per node) | STANDARD |
| `MIN/MAX_CLUSTER_COUNT` | Multi-cluster warehouse scaling (Enterprise+) | 1/1 |
| `SCALING_POLICY` | `STANDARD` (favor performance) or `ECONOMY` (favor cost) | STANDARD |

### Key Principles
- Warehouses are **independent** — one warehouse's load doesn't affect another
- **Scaling up** (bigger size) = faster complex queries
- **Scaling out** (more clusters) = higher concurrency
- Credit consumption: XSmall = 1 credit/hr, Small = 2, Medium = 4, Large = 8, etc.
- Warehouses **do not store data** — they are pure compute

## 1.5 Micro-Partitions & Storage

- Data is stored in **columnar, compressed, encrypted micro-partitions** (50–500 MB)
- Snowflake manages partitioning automatically (no user-defined partitions)
- Each micro-partition has metadata: min/max values, distinct count, null count per column
- This metadata enables **partition pruning** — skipping irrelevant partitions at query time
- **Clustering keys** (Enterprise+) can reorder data for better pruning on specific columns

### Table Types

| Type | Time Travel | Fail-safe | Use Case |
|------|-------------|-----------|----------|
| **Permanent** | Up to 90 days (Enterprise) | 7 days | Production data |
| **Transient** | Up to 1 day | None | ETL staging, intermediate data |
| **Temporary** | Up to 1 day | None | Session-scoped; dropped at session end |
| **External** | No | No | Query data in cloud storage without loading |
| **Dynamic** | Yes | Yes | Declarative transformation pipelines **(V2)** |

## 1.6 Caching Layers

Snowflake has **three** caching layers:

| Cache | Where | TTL | Triggered By |
|-------|-------|-----|-------------|
| **Query Result Cache** | Cloud Services layer | 24 hours (resets on re-use) | Exact same query, same role, underlying data unchanged |
| **Metadata Cache** | Cloud Services layer | Always on | `COUNT(*)`, `MIN/MAX`, `SHOW TABLES` — answered from micro-partition metadata |
| **Warehouse (Local Disk) Cache** | Compute layer (SSD) | Until warehouse suspends | Subsequent queries scanning same partitions benefit from warm SSD |

**Exam tip:** Result cache is free (Cloud Services). Warehouse cache requires the warehouse to be running. Suspending a warehouse clears its local disk cache. **(V1)**

## 1.7 Snowflake AI & ML Features **(V3)**

### Cortex AI SQL Functions
| Function | Purpose | Returns |
|----------|---------|--------|
| `SNOWFLAKE.CORTEX.SENTIMENT(text)` | Numeric sentiment score | FLOAT: -1 (negative) to +1 (positive) |
| `AI_CLASSIFY(text, labels)` | Classify text into categories | OBJECT with `:labels[0]` |
| `SNOWFLAKE.CORTEX.EXTRACT_ANSWER(text, question)` | Extract answer from text | ARRAY of answer objects |
| `AI_SUMMARIZE_AGG(text)` | Summarize grouped text rows | VARCHAR summary |
| `AI_COMPLETE(model, prompt)` | General-purpose LLM completion | VARCHAR |
| `AI_TRANSLATE(text, from, to)` | Translate text | VARCHAR |
| `AI_SENTIMENT(text [, categories])` | Categorical sentiment (pos/neg/neutral/mixed) | OBJECT |

### Other AI/ML Capabilities **(Gap)**
- **Cortex Search** — semantic search over text columns
- **Cortex Analyst** — natural-language queries over semantic views
- **Cortex Code** — agentic AI coding assistant in Snowsight **(V3-cortexcode)**
- **Snowflake ML** — model training, feature store, model registry
- **Snowpark ML** — Python ML library for training/inference
- **Snowflake Notebooks** — Jupyter-based interactive environment
- **Streamlit in Snowflake** — build data apps

**Exam tip:** Cortex AI functions require the `SNOWFLAKE.CORTEX_USER` database role.

---
# Domain 2.0 — Account Management & Governance (20%)

## 2.1 RBAC & System Roles **(V4)**

Snowflake uses **Role-Based Access Control (RBAC)** combined with **Discretionary Access Control (DAC)**.

### System Role Hierarchy
```
ORGADMIN
 └── ACCOUNTADMIN  (top account-level role — combines SYSADMIN + SECURITYADMIN)
      ├── SECURITYADMIN  (manages grants, roles, users)
      │    └── USERADMIN  (creates users and roles)
      └── SYSADMIN  (manages all database objects)
           └── custom roles (should be granted to SYSADMIN)
                └── PUBLIC  (every user has this by default)
```

### Key Principles
- **Least privilege** — grant only what's needed
- Every securable object has an **owner** (the role that created it)
- Privileges flow **upward** through the role hierarchy
- `ACCOUNTADMIN` should **not** be a default role; use it only when necessary
- Always grant custom roles **up to SYSADMIN** so objects remain accessible

**Exam tip:** `SECURITYADMIN` can grant privileges because it has `MANAGE GRANTS`. `USERADMIN` can create users/roles but cannot grant object privileges.

## 2.2 Privilege Grants **(V4)**

### Privilege Levels
| Object | Common Privileges |
|--------|-------------------|
| Warehouse | `USAGE`, `OPERATE`, `MONITOR`, `MODIFY` |
| Database | `USAGE`, `MONITOR`, `CREATE SCHEMA` |
| Schema | `USAGE`, `CREATE TABLE`, `CREATE VIEW`, `CREATE STAGE`, etc. |
| Table | `SELECT`, `INSERT`, `UPDATE`, `DELETE`, `TRUNCATE` |
| View | `SELECT` |

### Grant Patterns from Vignettes
```sql
-- Create custom role
CREATE OR REPLACE ROLE tb_data_steward;

-- Grant warehouse access
GRANT OPERATE, USAGE ON WAREHOUSE tb_dev_wh TO ROLE tb_data_steward;

-- Grant database/schema access
GRANT USAGE ON DATABASE tb_101 TO ROLE tb_data_steward;
GRANT USAGE ON ALL SCHEMAS IN DATABASE tb_101 TO ROLE tb_data_steward;

-- Grant table access
GRANT SELECT ON ALL TABLES IN SCHEMA raw_customer TO ROLE tb_data_steward;

-- Grant role to user (dynamic)
SET my_user = CURRENT_USER();
GRANT ROLE tb_data_steward TO USER IDENTIFIER($my_user);
```

**Exam tip:** `USAGE` on a database/schema doesn't grant access to objects inside — you need separate grants on tables/views. `ALL` as a privilege name grants every applicable privilege.

## 2.3 Tags & Data Classification **(V4)**

### Tags
- **Tags** are schema-level objects that attach key-value metadata to columns/tables
- Created with `CREATE TAG`, applied with `ALTER TABLE ... SET TAG`
- Can be queried via `INFORMATION_SCHEMA.TAG_REFERENCES_ALL_COLUMNS()`
- **Tag-based masking** — attach masking policies to tags instead of individual columns

### Auto-Classification (Enterprise+)
- `SNOWFLAKE.DATA_PRIVACY.CLASSIFICATION_PROFILE` — automatic, scheduled classification
- `SYSTEM$CLASSIFY()` — on-demand classification
- System tags applied: `SNOWFLAKE.CORE.SEMANTIC_CATEGORY`, `SNOWFLAKE.CORE.PRIVACY_CATEGORY`
- Semantic categories: `NAME`, `EMAIL`, `PHONE_NUMBER`, `DATE_OF_BIRTH`, `POSTAL_CODE`, etc.
- Privacy categories: `IDENTIFIER`, `QUASI_IDENTIFIER`, `SENSITIVE`

**Vignette lesson:** On a Standard/Trial account, none of the auto-classification features are available. You must tag columns manually. **(V4)**

## 2.4 Column-Level Security — Masking Policies **(V4)**

Dynamic Data Masking hides sensitive data at query time based on the querying role.

```sql
-- String masking policy
CREATE OR REPLACE MASKING POLICY governance.mask_string_pii
  AS (original_value STRING) RETURNS STRING ->
  CASE
    WHEN CURRENT_ROLE() NOT IN ('ACCOUNTADMIN', 'TB_ADMIN')
      THEN '****MASKED****'
    ELSE original_value
  END;

-- Date masking policy (truncates to year)
CREATE OR REPLACE MASKING POLICY governance.mask_date_pii
  AS (original_value DATE) RETURNS DATE ->
  CASE
    WHEN CURRENT_ROLE() NOT IN ('ACCOUNTADMIN', 'TB_ADMIN')
      THEN DATE_TRUNC('year', original_value)
    ELSE original_value
  END;

-- Attach masking policies to a TAG (tag-based masking)
ALTER TAG governance.pii SET
    MASKING POLICY governance.mask_string_pii,
    MASKING POLICY governance.mask_date_pii;
```

**Key concepts:**
- One masking policy per **data type per tag** (string policy + date policy on same tag is valid)
- Policy evaluates at query time — different roles see different results
- `CURRENT_ROLE()` is the standard function for conditional masking
- Tag-based masking automatically applies to any column carrying that tag

## 2.5 Row Access Policies **(V4)**

Row-level security filters which rows a role can see.

```sql
-- Mapping table: which role can see which country
CREATE TABLE governance.row_policy_map (
    role STRING, country_permission STRING
);
INSERT INTO governance.row_policy_map
    VALUES ('TB_DATA_ENGINEER', 'United States');

-- Row access policy
CREATE OR REPLACE ROW ACCESS POLICY governance.customer_loyalty_policy
    AS (country STRING) RETURNS BOOLEAN ->
        CURRENT_ROLE() IN ('ACCOUNTADMIN', 'SYSADMIN')
        OR EXISTS (
            SELECT 1 FROM governance.row_policy_map rp
            WHERE UPPER(rp.role) = CURRENT_ROLE()
              AND rp.country_permission = country
        );

-- Attach to table
ALTER TABLE raw_customer.customer_loyalty
    ADD ROW ACCESS POLICY governance.customer_loyalty_policy ON (country);
```

**Exam tip:** A row access policy returns BOOLEAN — TRUE = row is visible. Only **one** row access policy per table. Masking policies are per-column; row access policies are per-table.

## 2.6 Data Metric Functions (DMFs) **(V4)**

DMFs monitor data quality by running checks on table data.

### System DMFs
```sql
SELECT SNOWFLAKE.CORE.NULL_PERCENT(SELECT customer_id FROM raw_pos.order_header);
SELECT SNOWFLAKE.CORE.DUPLICATE_COUNT(SELECT order_id FROM raw_pos.order_header);
SELECT SNOWFLAKE.CORE.AVG(SELECT order_total FROM raw_pos.order_header);
```

### Custom DMFs
```sql
CREATE OR REPLACE DATA METRIC FUNCTION governance.invalid_order_total_count(
    order_prices_t TABLE(order_total NUMBER, unit_price NUMBER, quantity INTEGER)
) RETURNS NUMBER AS
'SELECT COUNT(*) FROM order_prices_t WHERE order_total != unit_price * quantity';

-- Schedule and attach
ALTER TABLE raw_pos.order_detail SET DATA_METRIC_SCHEDULE = 'TRIGGER_ON_CHANGES';
ALTER TABLE raw_pos.order_detail
    ADD DATA METRIC FUNCTION governance.invalid_order_total_count
    ON (price, unit_price, quantity);
```

**Exam tip:** DMFs use **table parameter syntax** `TABLE(col1 type, col2 type)`. Schedule options: `TRIGGER_ON_CHANGES`, or cron expressions.

## 2.7 Cost Management **(V1)**

### Resource Monitors
Credit-based spending controls that attach to warehouses.

```sql
CREATE OR REPLACE RESOURCE MONITOR my_resource_monitor
    WITH CREDIT_QUOTA = 100
    FREQUENCY = MONTHLY
    START_TIMESTAMP = IMMEDIATELY
    TRIGGERS
        ON 75 PERCENT DO NOTIFY
        ON 90 PERCENT DO SUSPEND
        ON 100 PERCENT DO SUSPEND_IMMEDIATE;

ALTER WAREHOUSE my_wh SET RESOURCE_MONITOR = my_resource_monitor;
```

### Budgets
`SNOWFLAKE.CORE.BUDGET` — dollar-amount spending limits across any Snowflake service (not just warehouses). **(V1)**

**Exam tip:** `SUSPEND` lets running queries finish; `SUSPEND_IMMEDIATE` kills them. Resource monitors track **credits**, budgets track **dollars**. Only `ACCOUNTADMIN` can create account-level resource monitors.

## 2.8 Additional Governance Concepts **(Gap)**

| Concept | Key Points |
|---------|------------|
| **Network Policies** | IP allowlist/blocklist; can be set at account or user level |
| **MFA** | Multi-factor auth; Snowflake recommends for all human users |
| **Key-Pair Authentication** | For programmatic access (service accounts) |
| **SSO / SAML / OAuth** | Federated identity; SCIM for user/role provisioning |
| **Alerts** | Monitor conditions and trigger notifications |
| **ACCOUNT_USAGE schema** | Latent (45 min–3 hr) views for login history, query history, storage, etc. |
| **Trust Center** | Scanner packages for security posture: Security Essentials, CIS Benchmarks, Threat Intelligence **(V4)** |

---
# Domain 3.0 — Data Loading & Connectivity (18%)

## 3.1 Stages **(V2)**

Stages are named locations where data files sit before/after loading.

| Stage Type | Location | Syntax |
|-----------|----------|--------|
| **User stage** | Each user has one automatically (`@~`) | `PUT file:// @~` |
| **Table stage** | Each table has one automatically (`@%table`) | `COPY INTO table FROM @%table` |
| **Named internal stage** | Schema-level object, shared across users | `CREATE STAGE my_stage` |
| **Named external stage** | Points to S3/Azure Blob/GCS | `CREATE STAGE my_stage URL='s3://...'` |

```sql
-- External stage pointing to S3 (V2)
CREATE OR REPLACE STAGE raw_pos.menu_stage
    COMMENT = 'Stage for menu data'
    URL = 's3://sfquickstarts/frostbyte_tastybytes/raw_pos/menu/'
    FILE_FORMAT = public.csv_ff;
```

**Exam tip:** External stages require a **storage integration** for authentication (IAM role for AWS, service principal for Azure). The vignette used a public bucket so no integration was needed.

## 3.2 File Formats & COPY INTO **(V2)**

### File Formats
Reusable objects defining how to parse files: CSV, JSON, Avro, ORC, Parquet, XML.

```sql
CREATE OR REPLACE FILE FORMAT public.csv_ff
    TYPE = 'CSV'
    FIELD_DELIMITER = ','
    SKIP_HEADER = 1
    FIELD_OPTIONALLY_ENCLOSED_BY = '"';
```

### COPY INTO (Loading)
```sql
COPY INTO raw_pos.menu_staging FROM @raw_pos.menu_stage;
```

**Key options:**
- `ON_ERROR = CONTINUE | SKIP_FILE | ABORT_STATEMENT` — error handling
- `VALIDATION_MODE = RETURN_ERRORS | RETURN_n_ROWS` — dry run
- `FORCE = TRUE` — reload previously loaded files
- `MATCH_BY_COLUMN_NAME` — for semi-structured formats
- `PURGE = TRUE` — delete files after successful load

### COPY INTO (Unloading) **(Gap)**
```sql
COPY INTO @my_stage/export/ FROM my_table
    FILE_FORMAT = (TYPE = 'PARQUET');
```

**Exam tip:** COPY INTO tracks loaded files for 64 days to prevent re-loading. Use `FORCE = TRUE` to override.

## 3.3 Semi-Structured Data — VARIANT **(V2)**

### Navigation Syntax
| Syntax | Purpose | Example |
|--------|---------|--------|
| `:` (colon) | Access object key | `col:name` |
| `[]` (brackets) | Access array element | `col:items[0]` |
| `::` (cast) | Cast VARIANT to type | `col:price::NUMBER` |
| `CAST()` | Explicit cast | `CAST(col:price AS NUMBER)` |

### LATERAL FLATTEN
Explodes arrays/objects into rows — critical for normalizing semi-structured data.

```sql
-- Flatten an array from a VARIANT column (V2)
SELECT
    i.value::STRING AS ingredient_name,
    m.menu_item_health_metrics_obj:menu_item_id::INTEGER AS menu_item_id
FROM raw_pos.menu_staging m,
     LATERAL FLATTEN(
         INPUT => m.menu_item_health_metrics_obj
                   :menu_item_health_metrics[0]
                   :ingredients::ARRAY
     ) i;
```

**Exam tip:** `FLATTEN` produces columns: `SEQ`, `KEY`, `PATH`, `INDEX`, `VALUE`, `THIS`. The `VALUE` column holds each element. Always use `LATERAL` with `FLATTEN`.

## 3.4 Dynamic Tables **(V2)**

Declarative transformation tables that auto-refresh when upstream data changes.

```sql
CREATE OR REPLACE DYNAMIC TABLE harmonized.ingredient
    LAG = '1 minute'          -- target freshness
    WAREHOUSE = 'TB_DE_WH'    -- compute for refresh
AS
    SELECT DISTINCT
        i.value::STRING AS ingredient
    FROM raw_pos.menu_staging m,
         LATERAL FLATTEN(...) i;
```

### Key Properties
- **`LAG`** (target lag) — how stale the data can be before a refresh triggers
- **`WAREHOUSE`** — which warehouse runs the refresh
- Chain Dynamic Tables into a **DAG** (Directed Acyclic Graph) — downstream refreshes automatically
- Can be **incremental** if the query pattern allows it (no full recompute)
- Visualizable in Snowsight's **Graph** tab

**Exam tip:** Dynamic Tables replace the manual pattern of streams + tasks for many pipeline use cases. They are declarative (you define the query, Snowflake handles scheduling).

## 3.5 Other Data Loading Concepts **(Gap)**

| Concept | Key Points |
|---------|------------|
| **Snowpipe** | Continuous, serverless loading triggered by cloud event notifications (SQS, Event Grid, Pub/Sub) |
| **Snowpipe Streaming** | Low-latency row-level ingestion via Snowflake Ingest SDK |
| **Streams** | Change Data Capture (CDC) — track INSERT, UPDATE, DELETE on a table |
| **Tasks** | Scheduled SQL execution (cron or interval); can form DAGs with predecessors |
| **Connectors/Drivers** | JDBC, ODBC, Python, Node.js, Go, .NET, Spark, Kafka |
| **Storage Integration** | Secure credential management for external stages (IAM role / service principal) |
| **Directory Tables** | Metadata catalog of files on a stage; auto-refresh capable |

---
# Domain 4.0 — Performance & Transformation (21%)

## 4.1 Query Performance Optimization

### Query Profile **(Gap)**
The **Query Profile** in Snowsight is your primary diagnostic tool. Key things to look for:

| Indicator | Problem | Fix |
|-----------|---------|-----|
| **Spilling to local/remote storage** | Warehouse too small for data volume | Scale up warehouse size |
| **Exploding joins** | Missing/incorrect join condition | Fix join predicates |
| **Full table scans (low pruning %)** | No useful filter predicates | Add WHERE clauses, clustering keys |
| **Queuing** | Too many concurrent queries | Scale out (multi-cluster) or stagger queries |
| **Large result sets** | Returning too much data to client | Use LIMIT, aggregate, or filter |

### Performance Features

| Feature | Edition | Purpose |
|---------|---------|--------|
| **Clustering Keys** | Enterprise+ | Reorder micro-partitions for better pruning on frequently filtered columns |
| **Search Optimization Service (SOS)** | Enterprise+ | Accelerates point lookups and substring searches |
| **Query Acceleration Service (QAS)** | Enterprise+ | Offloads scan-heavy portions of queries to serverless compute |
| **Materialized Views** | Enterprise+ | Pre-computed query results; auto-maintained by Snowflake |

**Exam tip:** Clustering keys are best for large tables (multi-TB) with known filter patterns. They are NOT indexes — they reorganize data within micro-partitions.

## 4.2 Query Result Cache Deep Dive **(V1)**

### When the cache is used:
1. Same query text (exact match)
2. Same role executing the query
3. Underlying data has **not** changed
4. Micro-partition metadata has not changed
5. Query does not use non-deterministic functions (`CURRENT_TIMESTAMP()`, `RANDOM()`, etc.)
6. Result cache has not expired (24 hours, resets on hit)

### When the cache is NOT used:
- Different role or different warehouse (role matters, warehouse doesn't)
- Underlying table data changed (DML, merge, etc.)
- Query uses `CURRENT_DATE()`, `RANDOM()`, `UUID_STRING()`, external functions
- Cache expired

**Exam tip:** Cache hits are **free** — no warehouse needed, no credits consumed. The warehouse doesn't even need to be running.

## 4.3 Key Transformation Functions

### From the Vignettes

| Function | Purpose | Source |
|----------|---------|--------|
| `LATERAL FLATTEN(INPUT => ...)` | Explode arrays/objects into rows | V2 |
| `CAST(expr AS type)` / `expr::type` | Type conversion | V2 |
| `ARRAY_AGG(col)` | Aggregate values into an array | V2 |
| `ARRAY_SIZE(array)` | Count elements in an array | V3-CC |
| `PARSE_JSON(string)` | Parse JSON string into VARIANT | V2 |
| `DATEDIFF('day', start, end)` | Date difference | V3-CC |
| `DATE_TRUNC('year', date)` | Truncate date to period | V4 |
| `EXTRACT(YEAR FROM ts)` / `MONTH(ts)` | Extract date parts | V2 |
| `ROUND(expr, n)` | Round to n decimal places | V3-CC |
| `ILIKE` | Case-insensitive LIKE | V3 |
| `LEFT(text, n)` | Substring from left | V3 |
| `LENGTH(text)` | String length | V3 |
| `string1 \|\| string2` | String concatenation | V3-CC |
| `* EXCLUDE (col1, col2)` | Select all columns except named ones | V1 |
| `CURRENT_ROLE()` | Current session role | V4 |
| `IDENTIFIER($var)` | Dynamic object reference from session variable | V4 |

### Window Functions **(Gap)**
```sql
-- Common exam patterns
ROW_NUMBER() OVER (PARTITION BY dept ORDER BY salary DESC)
RANK() OVER (ORDER BY score DESC)
LAG(col, 1) OVER (ORDER BY date)
SUM(amount) OVER (PARTITION BY region ORDER BY month ROWS UNBOUNDED PRECEDING)
```

**Exam tip:** Snowflake supports `QUALIFY` — filter on window function results without a subquery:
```sql
SELECT * FROM employees
QUALIFY ROW_NUMBER() OVER (PARTITION BY dept ORDER BY salary DESC) = 1;
```

## 4.4 Stored Procedures & UDFs **(Gap)**

| Object | Returns | Side Effects | Languages |
|--------|---------|-------------|----------|
| **UDF** (User-Defined Function) | Scalar or tabular value | No (read-only) | SQL, JavaScript, Python, Java, Scala |
| **UDTF** (Table Function) | Table (rows) | No | Same as UDF |
| **Stored Procedure** | Single value | Yes (DDL/DML allowed) | SQL, JavaScript, Python, Java, Scala |

**Exam tip:** UDFs can be used in `SELECT`, `WHERE`, `JOIN`. Procedures are called with `CALL`. Procedures run with **caller's rights** or **owner's rights**.

---
# Domain 5.0 — Data Collaboration (10%)

## 5.1 Zero-Copy Cloning **(V1)**

Instant copy that shares the same underlying micro-partitions until data diverges.

```sql
-- Clone a table
CREATE OR REPLACE TABLE raw_pos.truck_dev CLONE raw_pos.truck_details;

-- Clone an entire database or schema
CREATE DATABASE dev_db CLONE prod_db;
```

### Key Properties
- **No additional storage** until data is modified (then only changed partitions are duplicated)
- Works on tables, schemas, and databases
- Cloned object is **independent** — changes don't affect the source
- Metadata (grants, tags, policies) can optionally be cloned
- Commonly used for dev/test environments and safe experimentation

**Exam tip:** Cloning is a **metadata-only** operation — it's instant regardless of table size. Transient tables can be cloned; temporary tables cannot be cloned at the schema/database level.

## 5.2 Time Travel & Data Recovery **(V1)**

### Time Travel
Access historical data within the retention period.

```sql
-- Query data as of 5 minutes ago
SELECT * FROM my_table AT(OFFSET => -60*5);

-- Query data as of a specific timestamp
SELECT * FROM my_table AT(TIMESTAMP => '2024-01-15 10:00:00'::TIMESTAMP);

-- Query data before a specific statement
SELECT * FROM my_table BEFORE(STATEMENT => '<query_id>');

-- Restore a dropped table
DROP TABLE raw_pos.truck_details;
UNDROP TABLE raw_pos.truck_details;
```

### Retention Periods
| Edition | Permanent Tables | Transient/Temporary |
|---------|-----------------|---------------------|
| Standard | 0–1 day | 0–1 day |
| Enterprise+ | 0–90 days | 0–1 day |

### Table SWAP **(V1)**
```sql
-- Atomically swap two tables' content and metadata
ALTER TABLE raw_pos.truck_details SWAP WITH raw_pos.truck_dev;
```
Useful for blue-green deployments: build new data in a staging table, then swap.

## 5.3 Fail-Safe **(Gap)**

- **7-day** non-configurable recovery period **after** Time Travel expires
- Only Snowflake Support can recover data from Fail-safe
- Applies to **permanent** tables only (not transient or temporary)
- Consumes storage

```
|--- Time Travel (configurable) ---|--- Fail-safe (7 days, Snowflake only) ---|
```

**Exam tip:** Fail-safe is NOT user-accessible. It's a disaster recovery mechanism managed by Snowflake. Transient tables have NO Fail-safe — that's why they cost less storage.

## 5.4 Secure Data Sharing **(Gap)**

### Direct Sharing (Provider → Consumer)
- **No data movement** — consumers query the provider's data in place
- Provider creates a **Share** object, grants access to databases/schemas/tables
- Consumer creates a **database from share** in their account
- Consumer sees data in **read-only** mode
- **Real-time** — consumers always see the latest data

### Reader Accounts
- Created by the provider for consumers who don't have a Snowflake account
- Provider pays for the reader account's compute

### Marketplace
- **Public listings** — available to all Snowflake accounts
- **Private listings** — shared with specific accounts
- **Personalized listings** — customized per consumer
- Listings can contain: data shares, Native Apps, connected apps

### Data Clean Rooms
- Secure multi-party computation without exposing raw data
- Each party keeps their data in their own account

### Replication & Failover **(Gap)**
- **Database replication** — replicate databases across accounts/regions
- **Account replication** — replicate account-level objects (users, roles, warehouses)
- **Failover groups** — automatic failover to a secondary account for business continuity

**Exam tip:** Data sharing has **zero copy** — no data is transferred or duplicated. The consumer queries the provider's micro-partitions directly. Sharing is supported across regions and cloud providers.

---
# Practice Questions (25)
*Weighted by domain. Scroll past all questions for the answer key.*

---
## Domain 1 — Features & Architecture (8 questions)

**Q1.** A data analyst runs a `SELECT *` query on a large table. The warehouse is suspended. The query returns results in under 1 second with zero credits consumed. Which Snowflake feature is responsible?

A) Warehouse local disk cache  
B) Query result cache  
C) Metadata cache  
D) Materialized view  

---

**Q2.** A company needs to use dynamic data masking, row access policies, and multi-cluster virtual warehouses. What is the **minimum** Snowflake edition required?

A) Standard  
B) Enterprise  
C) Business Critical  
D) VPS  

---

**Q3.** Which Snowflake architectural layer is responsible for authentication, query optimization, and metadata management?

A) Storage layer  
B) Compute layer  
C) Cloud Services layer  
D) Network layer  

---

**Q4.** A warehouse is sized as X-Large (16 credits/hr). A query is running slowly due to data spilling to remote storage. Which action would most likely improve performance?

A) Add a clustering key to the table  
B) Scale out by adding more clusters  
C) Scale up by increasing the warehouse size  
D) Enable the query result cache  

---

**Q5.** Which statement about virtual warehouses is TRUE?

A) A warehouse stores a copy of the data it queries  
B) Suspending a warehouse clears its local disk cache  
C) Two warehouses cannot query the same table concurrently  
D) Multi-cluster warehouses are available in Standard edition  

---

**Q6.** A Cortex AI function call `SNOWFLAKE.CORTEX.SENTIMENT(review)` returns a value of 0.82. What does this indicate?

A) The review is 82% complete  
B) The review has strong positive sentiment  
C) The review was classified into category 82  
D) The review is 82 tokens long  

---

**Q7.** In Snowflake's architecture, micro-partitions store metadata including min/max values per column. This metadata primarily enables which optimization?

A) Query result caching  
B) Partition pruning  
C) Data replication  
D) Automatic clustering  

---

**Q8.** A data engineer creates a `SNOWPARK_OPTIMIZED` warehouse. How does this differ from a `STANDARD` warehouse?

A) It costs fewer credits per hour  
B) It provides 16x more memory per node  
C) It can run SQL but not Python  
D) It automatically scales to more clusters  

## Domain 2 — Account Management & Governance (5 questions)

**Q9.** A masking policy uses `CURRENT_ROLE()` to determine whether to mask data. A user with the `ANALYST` role queries a column with the policy. What happens?

A) The query fails with an access error  
B) The policy evaluates and returns masked or unmasked data based on the CASE logic  
C) The user sees the raw data because masking only applies to `PUBLIC` role  
D) The column is dropped from the result set  

---

**Q10.** Which system role has the `MANAGE GRANTS` privilege, allowing it to grant privileges on any object to any role?

A) SYSADMIN  
B) USERADMIN  
C) SECURITYADMIN  
D) ACCOUNTADMIN  

---

**Q11.** A resource monitor is configured with `ON 90 PERCENT DO SUSPEND` and `ON 100 PERCENT DO SUSPEND_IMMEDIATE`. What is the difference between `SUSPEND` and `SUSPEND_IMMEDIATE`?

A) SUSPEND stops new queries; SUSPEND_IMMEDIATE also kills running queries  
B) SUSPEND sends a notification; SUSPEND_IMMEDIATE suspends the warehouse  
C) There is no difference; both suspend the warehouse immediately  
D) SUSPEND applies to the current warehouse; SUSPEND_IMMEDIATE applies to all warehouses  

---

**Q12.** A data steward needs to ensure that any column tagged with `PII` is automatically masked. Which approach achieves this?

A) Create a masking policy and attach it directly to each PII column  
B) Create a masking policy and attach it to the `PII` tag using `ALTER TAG`  
C) Create a row access policy on the table  
D) Set the column's `DEFAULT` value to `'****MASKED****'`  

---

**Q13.** A row access policy returns `FALSE` for a specific row and role combination. What happens when that role queries the table?

A) The query fails with an authorization error  
B) The row is excluded from the result set (not visible)  
C) The row is visible but all columns are masked  
D) The row is visible but read-only  

## Domain 3 — Data Loading & Connectivity (5 questions)

**Q14.** A data engineer uses `COPY INTO my_table FROM @my_stage` to load a CSV file. The next day, they run the exact same command. What happens?

A) The data is loaded again, creating duplicates  
B) The command succeeds but loads zero rows (file already tracked)  
C) The command fails with an error  
D) The data is loaded only if the file was modified since the last load  

---

**Q15.** Which function is used to convert a JSON string literal into a Snowflake VARIANT value?

A) `TO_JSON()`  
B) `JSON_PARSE()`  
C) `PARSE_JSON()`  
D) `CAST(string AS VARIANT)`  

---

**Q16.** A Dynamic Table is created with `LAG = '5 minutes'`. What does this mean?

A) The table refreshes exactly every 5 minutes on a fixed schedule  
B) The table data can be at most 5 minutes stale compared to its source  
C) The table retains 5 minutes of Time Travel history  
D) Query results from the table are cached for 5 minutes  

---

**Q17.** When using `LATERAL FLATTEN` on a VARIANT array, which output column contains the individual array elements?

A) `INDEX`  
B) `KEY`  
C) `VALUE`  
D) `THIS`  

---

**Q18.** What is the primary advantage of Snowpipe over scheduled `COPY INTO` statements?

A) Snowpipe is free of charge  
B) Snowpipe loads data continuously as files arrive, using serverless compute  
C) Snowpipe can load data from any URL without a stage  
D) Snowpipe automatically creates the target table if it doesn't exist  

## Domain 4 — Performance & Transformation (5 questions)

**Q19.** A query has a Query Profile showing "Bytes spilled to remote storage." What is the most effective fix?

A) Add a clustering key  
B) Use a larger warehouse  
C) Enable query acceleration service  
D) Add a `LIMIT` clause  

---

**Q20.** Which Snowflake feature allows you to filter on window function results directly, without wrapping the query in a subquery?

A) `HAVING`  
B) `QUALIFY`  
C) `FILTER`  
D) `WHERE ... OVER`  

---

**Q21.** Which of the following queries will NOT use the query result cache, even if the same query was run 5 minutes ago by the same role?

A) `SELECT COUNT(*) FROM large_table`  
B) `SELECT * FROM large_table WHERE id = 42`  
C) `SELECT CURRENT_TIMESTAMP(), * FROM large_table`  
D) `SELECT name, salary FROM large_table ORDER BY salary DESC LIMIT 10`  

---

**Q22.** The Snowflake `SELECT * EXCLUDE (col1, col2)` syntax is used to:

A) Delete columns col1 and col2 from the table  
B) Select all columns except col1 and col2  
C) Apply a masking policy to col1 and col2  
D) Filter out rows where col1 or col2 is NULL  

---

**Q23.** A data engineer needs a pre-computed, automatically maintained summary table that refreshes when the source data changes. Which feature should they use?

A) Temporary table with a task  
B) Materialized view  
C) External table  
D) Transient table with a stream  

## Domain 5 — Data Collaboration (2 questions)

**Q24.** A company clones a 5 TB production database for development testing. How much additional storage does the clone consume immediately?

A) 5 TB (full copy)  
B) Approximately 2.5 TB (compressed copy)  
C) Zero additional storage  
D) Storage depends on the warehouse size used for cloning  

---

**Q25.** A provider shares data with a consumer account in a different region using Secure Data Sharing. Which statement is TRUE?

A) The data is physically copied to the consumer's region  
B) The consumer must use a reader account  
C) Cross-region sharing requires database replication or auto-fulfillment  
D) Cross-region sharing is not supported  

---
# Answer Key

| # | Answer | Explanation |
|---|--------|-------------|
| **Q1** | **B** | Query result cache sits in the Cloud Services layer and can serve results without resuming a warehouse. The warehouse was suspended, so local disk cache (A) is unavailable. Metadata cache (C) only handles COUNT/MIN/MAX on metadata. |
| **Q2** | **B** | Dynamic data masking, row access policies, and multi-cluster warehouses all require Enterprise Edition. Business Critical adds compliance features but isn't the *minimum* needed. |
| **Q3** | **C** | Cloud Services handles authentication, query parsing/optimization, metadata management, access control, and the result cache. |
| **Q4** | **C** | Spilling to remote storage means the warehouse doesn't have enough memory/SSD for the operation. Scaling **up** (larger size = more memory) directly addresses this. Scaling out (B) helps concurrency, not single-query memory. |
| **Q5** | **B** | Suspending a warehouse deallocates its compute nodes, clearing the local SSD cache. Warehouses do NOT store data (A is false). Multiple warehouses can query the same table (C is false). Multi-cluster requires Enterprise (D is false). |
| **Q6** | **B** | `SNOWFLAKE.CORTEX.SENTIMENT()` returns a float from -1 to +1. Values ≥ 0.5 indicate positive sentiment. 0.82 is strong positive. |
| **Q7** | **B** | Min/max metadata per micro-partition enables partition pruning — Snowflake can skip partitions that don't contain relevant data based on filter predicates. |
| **Q8** | **B** | SNOWPARK_OPTIMIZED warehouses provide 16x more memory per node, designed for memory-intensive operations like large Snowpark DataFrames, ML training, and UDTFs. |
| **Q9** | **B** | Masking policies evaluate at query time using `CURRENT_ROLE()` in CASE logic. The query succeeds; the data is either masked or shown depending on the role. |
| **Q10** | **C** | SECURITYADMIN inherits the `MANAGE GRANTS` global privilege, enabling it to grant/revoke privileges on any object. ACCOUNTADMIN (D) also has this via inheritance, but SECURITYADMIN is the designated role for this purpose. |
| **Q11** | **A** | SUSPEND allows currently running queries to complete, then suspends. SUSPEND_IMMEDIATE cancels all running queries immediately, then suspends. |
| **Q12** | **B** | Tag-based masking attaches a policy to the tag itself (`ALTER TAG pii SET MASKING POLICY ...`). Any column carrying that tag is automatically masked — no per-column attachment needed. |
| **Q13** | **B** | Row access policies return BOOLEAN. FALSE means the row is silently excluded from results — no error, no visibility, just filtered out. |
| **Q14** | **B** | COPY INTO tracks loaded files for 64 days. Re-running the same command skips already-loaded files and reports 0 rows loaded. Use `FORCE = TRUE` to override. |
| **Q15** | **C** | `PARSE_JSON()` is the Snowflake function that converts a JSON string into a VARIANT value. `TO_JSON()` does the reverse (VARIANT → string). |
| **Q16** | **B** | LAG (target lag) defines the maximum allowed staleness. Snowflake schedules refreshes to keep the data within this freshness target. It's not a fixed interval. |
| **Q17** | **C** | The `VALUE` column in FLATTEN output contains each individual element. `INDEX` is the array position, `KEY` is for objects, `THIS` is the original input. |
| **Q18** | **B** | Snowpipe provides continuous, event-driven loading using serverless compute. Files are loaded within minutes of arriving on the stage, triggered by cloud notifications. |
| **Q19** | **B** | Spilling means the warehouse ran out of memory and SSD. A larger warehouse provides more resources to process the data in-memory. Clustering (A) helps pruning, not spilling. QAS (C) helps scan-heavy queries but doesn't solve memory spills. |
| **Q20** | **B** | `QUALIFY` is Snowflake's clause for filtering window function results directly. It eliminates the need for a subquery or CTE wrapper. |
| **Q21** | **C** | `CURRENT_TIMESTAMP()` is non-deterministic — its value changes with each execution. Queries with non-deterministic functions bypass the result cache. |
| **Q22** | **B** | `EXCLUDE` is a Snowflake SQL extension that selects all columns except the listed ones. It does not modify the table structure or filter rows. |
| **Q23** | **B** | Materialized views are pre-computed, automatically refreshed by Snowflake when source data changes. They serve as persistent query result caches with guaranteed freshness. |
| **Q24** | **C** | Zero-copy cloning is a metadata operation. No data is physically copied. Storage is only consumed when data in the clone **diverges** from the source (new micro-partitions are created). |
| **Q25** | **C** | Direct Secure Data Sharing works within the same region. Cross-region sharing requires database replication or Snowflake's auto-fulfillment feature to replicate data to the consumer's region. |